# Lesson 13 Lab — N:M Semi-structured Sparsity and the 2:4 Contract

**Puzzle:** Does a tensor with 50% zeros qualify for Sparse Tensor Core execution?

This notebook is designed for a CUDA GPU and retains the output of a complete RTX 5090 run.


## Why this matters

NVIDIA's 2:4 path imposes a local pattern: within each group of four values along the required dimension, at least two are zero and the representation must be compressed for a supported sparse GEMM. Global sparsity, pattern compliance, backend conversion, and selected tactic are separate gates.


## 0. Predict before running

1. Predict the compliance of a random 50% mask.
2. Prove that top-2-of-4 masking reaches exactly 50% sparsity.
3. List the evidence needed after compliance before claiming acceleration.

For every answer, name the observation that would prove it wrong.


## 1. Name the concrete objects

A BF16 weight matrix, a random 50% mask, a magnitude-based exact 2:4 mask, a local compliance checker, ordinary dense timing, and an optional PyTorch semi-structured conversion probe are recorded.

- 2:4 is checked per local group, not over the whole tensor.
- Pattern compliance precedes backend compression and tactic selection.
- Dense-path timing cannot establish sparse Tensor Core execution.


## 2. Derive the mechanism

Partition the contracted dimension into groups of four and retain the two largest magnitudes per group. This guarantees exactly 2 nonzeros per group while preserving 50% globally. A random half mask only satisfies a fraction of groups. Even a compliant tensor remains dense storage until converted into the backend's compressed format, and the hardware/library combination must support its shape and dtype.

### Mechanism at a glance

```mermaid
flowchart LR
  W["dense group<br/>w0 w1 w2 w3"] --> K["keep top two magnitudes"]
  K --> M["2:4 values<br/>two nonzeros + two zeros"]
  M --> C{"backend conversion<br/>supported?"}
  C -->|"yes"| S["compressed sparse operand"]
  C -->|"no"| D["ordinary dense storage/path"]
  S --> T["sparse tactic + matched benchmark"]
```

### Walk it step by step

1. **Group along the contracted axis.** Reshape the supported weight dimension into consecutive groups of four.
2. **Keep exactly two values.** Top-2 magnitude selection creates 50% global sparsity and 100% local 2:4 compliance.
3. **Convert to the backend representation.** A compliant dense tensor is not yet a cuSPARSELt or TensorRT sparse operand.
4. **Prove the selected tactic.** Validate output, capture the sparse operator or tactic, and compare with a matched dense baseline.


## 3. Verify the execution environment

Inspect the next cell before running it: it asserts CUDA, fixes the seed, defines transparent timing/numerical helpers, and prints the GPU/PyTorch/CUDA record needed to interpret every output.


In [1]:
LESSON_NO = 13
LESSON_TITLE = 'N:M Semi-structured Sparsity and the 2:4 Contract'

from pathlib import Path
import copy, gzip, hashlib, importlib.util, io, json, math, random, shutil, statistics, sys
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260808 + LESSON_NO
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = False

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name,
    "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__,
    "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0],
    "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered:
        return float("nan")
    position = (len(ordered) - 1) * q
    lo, hi = math.floor(position), math.ceil(position)
    if lo == hi:
        return ordered[lo]
    return ordered[lo] * (hi - position) + ordered[hi] * (position - lo)

def cuda_times(fn, warmup=6, repeats=24):
    with torch.inference_mode():
        for _ in range(warmup):
            fn()
        torch.cuda.synchronize()
        samples = []
        for _ in range(repeats):
            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)
            start.record()
            fn()
            end.record()
            end.synchronize()
            samples.append(float(start.elapsed_time(end)))
    return samples

def timing_summary(samples):
    return {
        "median_ms": float(statistics.median(samples)),
        "p95_ms": float(percentile(samples, 0.95)),
        "p99_ms": float(percentile(samples, 0.99)),
        "samples_ms": [float(x) for x in samples],
    }

def count_params(module):
    return int(sum(p.numel() for p in module.parameters()))

def zero_fraction(tensor):
    return float((tensor == 0).float().mean().item())

def magnitude_mask(tensor, sparsity):
    flat = tensor.detach().abs().flatten()
    prune_count = int(round(flat.numel() * float(sparsity)))
    prune_count = min(max(prune_count, 0), flat.numel())
    mask = torch.ones_like(flat)
    if prune_count:
        idx = torch.topk(flat, prune_count, largest=False).indices
        mask[idx] = 0
    return mask.view_as(tensor)

def exact_2_4_mask(weight):
    assert weight.shape[-1] % 4 == 0
    groups = weight.detach().abs().reshape(*weight.shape[:-1], -1, 4)
    keep = torch.topk(groups, 2, dim=-1, largest=True).indices
    mask = torch.zeros_like(groups)
    mask.scatter_(-1, keep, 1)
    return mask.reshape_as(weight)

def compliance_2_4(weight):
    groups = weight.detach().reshape(*weight.shape[:-1], -1, 4)
    return float(((groups != 0).sum(dim=-1) == 2).float().mean().item())

def tensor_metrics(reference, candidate):
    ref = reference.float()
    cand = candidate.float()
    delta = cand - ref
    return {
        "rmse": float(torch.sqrt(torch.mean(delta.square())).item()),
        "mae": float(torch.mean(delta.abs()).item()),
        "max_error": float(delta.abs().max().item()),
        "cosine": float(F.cosine_similarity(ref.flatten(), cand.flatten(), dim=0).item()),
    }

def spearman(a, b):
    a = torch.as_tensor(a, dtype=torch.float64)
    b = torch.as_tensor(b, dtype=torch.float64)
    ra = torch.empty_like(a)
    rb = torch.empty_like(b)
    ra[torch.argsort(a)] = torch.arange(a.numel(), dtype=torch.float64)
    rb[torch.argsort(b)] = torch.arange(b.numel(), dtype=torch.float64)
    ra -= ra.mean(); rb -= rb.mean()
    return float((ra @ rb / (ra.norm() * rb.norm() + 1e-12)).item())


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.12.0",
  "cuda_runtime": "13.0",
  "python": "3.12.13",
  "seed": 20260821
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | random global 50% sparsity executed through ordinary dense matmul |
| Candidate | exact magnitude 2:4 sparsity plus an optional native conversion probe |
| Held constant | source weights, shape, dtype, input, zero budget, GPU, and timing protocol |
| Measurements | global sparsity, local compliance, dense-path latency, conversion availability, and conversion error |
| Evidence | `compatibility-probe` |

**Experiment:** Compare random and exact 2:4 masks, then attempt a native semi-structured conversion without hiding incompatibility.


## 5. Read the experiment code

The compliance function reshapes the K dimension into groups of four and counts nonzeros. The native conversion attempt is wrapped and stores either a successful sparse result or the exact exception text. Ordinary dense timing remains a control and is never relabeled as a sparse-kernel benchmark.

Do not execute until the code implements the frozen table above.


In [2]:
dtype=torch.bfloat16; rows,cols,batch=1024,1024,32
w=torch.randn(rows,cols,device=DEVICE,dtype=dtype); x=torch.randn(batch,cols,device=DEVICE,dtype=dtype)
random_mask=(torch.rand_like(w.float())>0.5).to(dtype); random_w=w*random_mask; nm_w=w*exact_2_4_mask(w)
td=timing_summary(cuda_times(lambda:F.linear(x,w))); tn=timing_summary(cuda_times(lambda:F.linear(x,nm_w)))
native_ok=False; native_message="API unavailable"
try:
    if hasattr(torch.sparse,"to_sparse_semi_structured"):
        sparse=torch.sparse.to_sparse_semi_structured(nm_w); _=F.linear(x,sparse); torch.cuda.synchronize(); native_ok=True; native_message=str(type(sparse).__name__)
    else: native_message="torch.sparse.to_sparse_semi_structured is unavailable"
except Exception as exc:
    native_message=f"{type(exc).__name__}: {str(exc).splitlines()[0]}"
metrics={"random_compliance":compliance_2_4(random_w),"nm_compliance":compliance_2_4(nm_w),"random_sparsity":zero_fraction(random_w),"nm_sparsity":zero_fraction(nm_w),"dense_median_ms":td["median_ms"],"nm_dense_path_median_ms":tn["median_ms"],"native_conversion_succeeded":native_ok,"native_message":native_message,"shape":[rows,cols],"dtype":str(dtype)}
analysis=(f"Random 50% masking achieved {metrics['random_compliance']:.1%} local compliance, while top-2-of-4 reached "
          f"{metrics['nm_compliance']:.1%} at {metrics['nm_sparsity']:.1%} sparsity. Ordinary dense-path medians were "
          f"{td['median_ms']:.6f} and {tn['median_ms']:.6f} ms. Native semi-structured conversion success was {native_ok}; "
          f"the retained probe message is `{native_message}`.")


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| Random-mask compliance | 37.45% |
| 2:4 compliance | 100.00% |
| 2:4 sparsity | 50.00% |
| Dense baseline median | 0.018544 ms |
| 2:4 dense-path median | 0.018512 ms |
| Native conversion | no |


## 7. Interpret rather than merely print

Random 50% masking achieved 37.4% local compliance, while top-2-of-4 reached 100.0% at 50.0% sparsity. Ordinary dense-path medians were 0.018544 and 0.018512 ms. Native semi-structured conversion success was False; the retained probe message is `RuntimeError: cuSPARSELt not supported on your machine.`.

The result is bounded to the shapes, seed, packages, and evidence label printed here.


## 8. Keep the evidence label honest

This run is labeled **`compatibility-probe`**. The notebook records real package/API availability and preserves the native success or failure state. Missing backend execution remains unmeasured.

The next cell writes the canonical JSON artifact and prints the same payload.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 13,
    "title": 'N:M Semi-structured Sparsity and the 2:4 Contract',
    "environment": ENV,
    "evidence_label": 'compatibility-probe',
    "metrics": metrics,
    "analysis": analysis,
    "conclusion": 'Exact 2:4 values are a necessary data invariant; native representation and tactic evidence complete the execution claim.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 13,
  "title": "N:M Semi-structured Sparsity and the 2:4 Contract",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.12.0",
    "cuda_runtime": "13.0",
    "python": "3.12.13",
    "seed": 20260821
  },
  "evidence_label": "compatibility-probe",
  "metrics": {
    "random_compliance": 0.37445068359375,
    "nm_compliance": 1.0,
    "random_sparsity": 0.5001668930053711,
    "nm_sparsity": 0.5,
    "dense_median_ms": 0.018543999642133713,
    "nm_dense_path_median_ms": 0.018511999398469925,
    "native_conversion_succeeded": false,
    "native_message": "RuntimeError: cuSPARSELt not supported on your machine.",
    "shape": [
      1024,
      1024
    ],
    "dtype": "torch.bfloat16"
  },
  "analysis": "Random 50% masking achieved 37.4% local compliance, while top-2-of-4 reached 100.0% at 50.0% sparsity. Ordinary dense-path medians were 0.018544 and 0.018512 ms. Native semi-structured conversion success was False; 

## 9. Make the bounded decision

> Exact 2:4 values are a necessary data invariant; native representation and tactic evidence complete the execution claim.

**Acceptance/rollback:** Accept a 2:4 speed claim only after compliance, supported compression, a sparse operator trace, numerical validation, and a matched dense baseline all pass.

**Failure analysis:** Zeros can be arranged along the wrong axis, shapes can violate alignment, and a library can fall back to dense tactics. A failed PyTorch conversion on one stack does not imply the GPU lacks all 2:4 support; it bounds only that API path.


## 10. Extend the evidence

Run the same compliant weights through cuSPARSELt or TensorRT, retain build logs and kernel names, and sweep supported shapes and FP16/BF16/INT8 dtypes.

The full evidence boundary and references are in [`README.md`](README.md).
